In [5]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


np.random.seed(42)
dates = pd.date_range("2023-01-01", periods=8760, freq="h")

base = 1200
energy = []
for dt in dates:
    hour_factor = 1.0 + 0.05 * np.sin(2 * np.pi * dt.hour / 24)
    weekend_factor = 0.95 if dt.weekday() >= 5 else 1.0
    noise = np.random.normal(0, 30)
    energy.append(round(base * hour_factor * weekend_factor + noise, 2))

df = pd.DataFrame({"datetime": dates, "energy_kwh": energy})
df.to_csv("../data/datacenter_hourly_energy.csv", index=False)
print("Done", df.head())

Done              datetime  energy_kwh
0 2023-01-01 00:00:00     1154.90
1 2023-01-01 01:00:00     1150.60
2 2023-01-01 02:00:00     1187.93
3 2023-01-01 03:00:00     1226.00
4 2023-01-01 04:00:00     1182.34


In [6]:
np.random.seed(99)
dates = pd.date_range("2023-01-01", periods=8760, freq="h")

base = 800
energy = []
for dt in dates:
    if dt.weekday() >= 5:  # weekend
        hour_factor = 0.3
    elif 9 <= dt.hour <= 18:  # office hours
        hour_factor = 1.0 + 0.1 * np.sin(np.pi * (dt.hour - 9) / 9)
    elif 7 <= dt.hour <= 9 or 18 <= dt.hour <= 20:  # ramp up/down
        hour_factor = 0.6
    else:  # night
        hour_factor = 0.2
    noise = np.random.normal(0, 20)
    energy.append(round(base * hour_factor + noise, 2))

df = pd.DataFrame({"datetime": dates, "energy_kwh": energy})
df.to_csv("../data/mnc_hourly_energy.csv", index=False)
print("Done", df.head())

Done              datetime  energy_kwh
0 2023-01-01 00:00:00      237.15
1 2023-01-01 01:00:00      281.14
2 2023-01-01 02:00:00      245.67
3 2023-01-01 03:00:00      266.60
4 2023-01-01 04:00:00      236.91


In [7]:
for name in ["datacenter", "mnc"]:
    df = pd.read_csv(f"../data/{name}_hourly_energy.csv", parse_dates=["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)
    
    df["hour"] = df["datetime"].dt.hour
    df["day_of_week"] = df["datetime"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["prev_hour_energy"] = df["energy_kwh"].shift(1)
    df["rolling_3h_avg"] = df["energy_kwh"].rolling(3).mean()
    df["rolling_6h_avg"] = df["energy_kwh"].rolling(6).mean()
    df["target_next_hour"] = df["energy_kwh"].shift(-1)
    df = df.dropna().reset_index(drop=True)
    
    df.to_csv(f"../data/{name}_ml_ready.csv", index=False)
    print(f"{name} done! Shape: {df.shape}")

datacenter done! Shape: (8754, 9)
mnc done! Shape: (8754, 9)


In [8]:
for name in ["datacenter", "mnc"]:
    df = pd.read_csv(f"../data/{name}_ml_ready.csv")
    
    features = ["hour", "day_of_week", "is_weekend", 
                "prev_hour_energy", "rolling_3h_avg", "rolling_6h_avg"]
    
    X = df[features]
    y = df["target_next_hour"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    score = model.score(X_test, y_test)
    print(f"{name} R² score: {score:.4f}")
    
    joblib.dump(model, f"../model/{name}_energy_rf.pkl")
    print(f"{name} model saved")

datacenter R² score: 0.6964
datacenter model saved
mnc R² score: 0.9941
mnc model saved
